In [ ]:
import sys, warnings
sys.path.insert(0, 'src')
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import trimesh
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({
    'figure.dpi':       150,
    'savefig.dpi':      200,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'lines.linewidth':  2.0,
    'lines.markersize': 5,
})

MATLAB_CSV = Path('Matlab/variables_esfera1.csv')
STL_PATH   = Path('data/esfera/esfera1.stl')
SAVE_DIR   = Path('results/Plots/Validacion')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

GAMMA       = 1.4
MACH_LIST   = [2, 4, 6, 8, 10, 15, 20]   # curvas MNM
U_INFTY_DIR = np.array([0., 1., 0.])      # dirección del flujo en MATLAB

In [ ]:
# ── Cargar CSV MATLAB ────────────────────────────────────────────────────────
df_mat = pd.read_csv(MATLAB_CSV, header=0, usecols=[0])
df_mat.columns = ['cp_matlab']
cp_matlab = df_mat['cp_matlab'].values   # shape (24648,)  — 0 en sotavento

# ── Cargar STL y calcular normales ───────────────────────────────────────────
mesh   = trimesh.load(str(STL_PATH), force='mesh')
verts  = np.asarray(mesh.vertices, dtype=float)
faces  = np.asarray(mesh.faces,    dtype=int)
v0, v1, v2 = verts[faces[:,0]], verts[faces[:,1]], verts[faces[:,2]]
cross  = np.cross(v1-v0, v2-v0)
norms  = np.linalg.norm(cross, axis=1, keepdims=True)
tnorm  = cross / (norms + 1e-30)   # normales unitarias OUTWARD

print(f'  Caras STL : {len(faces):,}')
print(f'  Filas CSV : {len(cp_matlab):,}')

# ── Ángulo local: Stheta = dot(U_infty, tnorm) ──────────────────────────────
# Stheta < 0  →  barlovento (cp_matlab > 0)
# Stheta > 0  →  sotavento  (cp_matlab = 0)
Stheta = tnorm @ U_INFTY_DIR          # shape (n_faces,)

# cos(phi) = -Stheta  (phi = ángulo de incidencia desde estancamiento)
cos_phi = -Stheta

# Máscara barlovento
wind = cos_phi > 0
print(f'  Barlovento: {wind.sum():,}   Sotavento: {(~wind).sum():,}')

# Verificar que el CSV coincide en longitud con el STL
assert len(cp_matlab) == len(faces), "Desajuste CSV / STL"

In [ ]:
# ── Funciones analíticas de Cp ────────────────────────────────────────────────

def cp_newton(cos_phi_arr):
    """MN: Cp = 2·cos²(φ)  (barlovento); 0 sotavento."""
    return np.where(cos_phi_arr > 0, 2.0 * cos_phi_arr**2, 0.0)

def cp_max_mnm(Mach, gamma=1.4):
    """Cp de estancamiento MNM — Rayleigh-Pitot."""
    g   = gamma
    M   = Mach
    p2_p1    = (2*g*M**2 - (g-1)) / (g+1)
    M2_sq    = ((g-1)*M**2 + 2) / (2*g*M**2 - (g-1))
    p02_p2   = (1 + (g-1)/2 * M2_sq) ** (g/(g-1))
    p02_pinf = p2_p1 * p02_p2
    return (2 / (g * M**2)) * (p02_pinf - 1)

def cp_mnm(cos_phi_arr, Mach, gamma=1.4):
    """MNM: Cp = Cp_max(M)·cos²(φ)  (barlovento); 0 sotavento."""
    cpmax = cp_max_mnm(Mach, gamma)
    return np.where(cos_phi_arr > 0, cpmax * cos_phi_arr**2, 0.0)

# Verificar cp_max en el límite hipersónico (debe → 2)
print('cp_max MNM por Mach:')
for M in MACH_LIST:
    print(f'  M={M:5.1f}  →  Cp_max = {cp_max_mnm(M):.5f}')

In [ ]:
# ── Plot 1: Cp vs cos(φ) — distribución en cara ──────────────────────────────
#  MATLAB scatter + MN line + MNM lines
phi_smooth = np.linspace(0, 1, 300)
colors_mnm = plt.cm.plasma(np.linspace(0.1, 0.85, len(MACH_LIST)))

fig, ax = plt.subplots(figsize=(10, 6))

# MATLAB data  (sólo barlovento, sample para no saturar el plot)
idx_wind = np.where(wind)[0]
rng = np.random.default_rng(42)
sample = rng.choice(idx_wind, size=min(3000, len(idx_wind)), replace=False)
ax.scatter(cos_phi[sample], cp_matlab[sample],
           s=8, alpha=0.35, color='steelblue', zorder=2, label='MATLAB (Newton)')

# MN model line
ax.plot(phi_smooth, cp_newton(phi_smooth),
        'k-', lw=2.5, zorder=5, label='MN model: 2·cos²φ')

# MNM model lines
for i, M in enumerate(MACH_LIST):
    cpmax = cp_max_mnm(M)
    ax.plot(phi_smooth, cpmax * phi_smooth**2,
            '--', color=colors_mnm[i], lw=1.8, zorder=4,
            label=f'MNM  M={M}  (Cp_max={cpmax:.3f})')

ax.set_xlabel('cos φ  (φ = ángulo desde estancamiento)', fontsize=12)
ax.set_ylabel('Cp', fontsize=12)
ax.set_title('Validación esfera — Distribución de Cp\nMATLAB vs MN / MNM', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.05)
ax.set_ylim(-0.05, 2.1)
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'validacion_Cp_vs_cosphi.png')
print('  → validacion_Cp_vs_cosphi.png')
plt.show()

In [ ]:
# ── Plot 2: Cp vs θ (ángulo en grados) — convenio MATLAB ─────────────────────
#  θ = 0°  →  estancamiento  (Cp = 2)
#  θ = 90° →  ecuador         (Cp = 0)
phi_deg = np.linspace(0, 90, 300)
cos_arr = np.cos(np.deg2rad(phi_deg))

phi_data_deg = np.degrees(np.arccos(np.clip(cos_phi[idx_wind], 0, 1)))

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(phi_data_deg[rng.choice(len(idx_wind), size=min(3000, len(idx_wind)), replace=False)],
           cp_matlab[idx_wind][rng.choice(len(idx_wind), size=min(3000, len(idx_wind)), replace=False)],
           s=8, alpha=0.35, color='steelblue', zorder=2, label='MATLAB (Newton)')

ax.plot(phi_deg, cp_newton(cos_arr), 'k-', lw=2.5, zorder=5, label='MN model: 2·cos²φ')

for i, M in enumerate(MACH_LIST):
    cpmax = cp_max_mnm(M)
    ax.plot(phi_deg, cpmax * cos_arr**2,
            '--', color=colors_mnm[i], lw=1.8, zorder=4,
            label=f'MNM  M={M}')

ax.set_xlabel('φ  (°)  desde estancamiento', fontsize=12)
ax.set_ylabel('Cp', fontsize=12)
ax.set_title('Validación esfera — Cp vs ángulo de incidencia\nMATLAB vs MN / MNM', fontsize=13, fontweight='bold')
ax.set_xlim(-1, 91)
ax.set_ylim(-0.05, 2.1)
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'validacion_Cp_vs_angulo.png')
print('  → validacion_Cp_vs_angulo.png')
plt.show()

In [ ]:
# ── Plot 3: Error MN modelo vs MATLAB — por cara ─────────────────────────────
#  Muestra que nuestro código reproduce exactamente el dato MATLAB
cp_mn_model  = cp_newton(cos_phi)          # nuestro modelo, todas las caras
err_abs      = np.abs(cp_mn_model - cp_matlab)   # error absoluto por cara

print(f'  Error MN modelo vs MATLAB:')
print(f'    max  = {err_abs.max():.3e}')
print(f'    mean = {err_abs.mean():.3e}')
print(f'    (diferencias debidas sólo a precisión numérica)')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(err_abs[wind], bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.set_xlabel('|Cp_modelo − Cp_MATLAB|  (caras de barlovento)', fontsize=11)
ax.set_ylabel('N° caras', fontsize=11)
ax.set_title('Histograma de error numérico MN — modelo vs MATLAB', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'validacion_error_histogram.png')
print('  → validacion_error_histogram.png')
plt.show()

In [ ]:
# ── Plot 4: CD esfera vs Mach (modelo MNM) ───────────────────────────────────
#  Referencia analítica Newton: CD = 1  (con S_ref = área frontal πR²)
from MNM import solve_modified_newton_case
from MN  import solve_newton_case

# Geometría de la esfera
areas   = np.linalg.norm(cross, axis=1) / 2.0
centers = (v0 + v1 + v2) / 3.0
ext     = verts.max(0) - verts.min(0)
S_ref   = float(ext[0] * ext[2])    # igual que el pipeline
L_ref   = float(ext[1])
r_ref   = np.average(centers, axis=0, weights=areas)

eD = np.array([0., 1., 0.])   # drag = dirección flujo
eM = np.array([1., 0., 0.])
eL = np.cross(eM, eD); eL /= np.linalg.norm(eL)

MACH_PLOT = [1.5, 2, 3, 4, 6, 8, 10, 12, 15, 20]
CDs_mnm = []
for M in MACH_PLOT:
    r = solve_modified_newton_case(
        centers=centers, areas=areas, normals=tnorm,
        alpha_deg=0.0, Mach=float(M),
        S_ref=S_ref, L_ref=L_ref, r_ref=r_ref,
        eD=eD, eL=eL, eM=eM, gamma=GAMMA,
    )
    CDs_mnm.append(float(np.dot(r['CF_total'], eD)))

r_mn = solve_newton_case(
    centers=centers, areas=areas, normals=tnorm,
    alpha_deg=0.0, S_ref=S_ref, L_ref=L_ref, r_ref=r_ref,
    eD=eD, eL=eL, eM=eM,
)
CD_mn = float(np.dot(r_mn['CF_total'], eD))

print(f'  CD MN (independiente de M) = {CD_mn:.5f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(MACH_PLOT, CDs_mnm, 'o-', color='#E53935', lw=2.5, ms=7, label='MNM (modelo)')
ax.axhline(CD_mn, color='#1565C0', ls='--', lw=2, label=f'MN (modelo)  CD={CD_mn:.3f}')
ax.set_xlabel('M∞', fontsize=12)
ax.set_ylabel('CD', fontsize=12)
ax.set_title('CD esfera vs M∞ — validación modelo', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'validacion_CD_vs_Mach.png')
print('  → validacion_CD_vs_Mach.png')
plt.show()